# Volatility Engine (VE) — 手機一鍵執行 (Google Colab)

在 **iPhone / iPad 的 Safari** 開 [colab.research.google.com](https://colab.research.google.com)，
把這個 notebook 打開後，**由上往下**逐格按左邊的 ▶️ 執行即可。

流程：
1. 下載程式碼 + 安裝套件
2. 抓**真實台股 ETF / 權值股**資料（Colab 有網路，不像研究環境被擋）
3. 跑完整 VE（第一~三階段）
4. 圖表 + 報告直接顯示在手機畫面

> 若 repo 是私有的，clone 會失敗 —— 兩個做法：把 repo 設為 public，或在下面填入 GitHub token。

## 1. 下載程式碼 + 安裝套件

In [ ]:
# 私有 repo 才需要 token；public 留空即可。
GITHUB_TOKEN = ""  # 例如 "ghp_xxx"，或留空
REPO = "hugekitoto/test1"

url = f"https://{GITHUB_TOKEN + '@' if GITHUB_TOKEN else ''}github.com/{REPO}.git"
import os
if not os.path.isdir("test1"):
    !git clone -q --branch claude/app-development-i0ualv $url
else:
    !cd test1 && git pull -q
%cd test1/volatility-engine
!pip install -q -r requirements.txt yfinance
print("\n✅ 準備完成")

## 2. 抓真實台股資料

改 `TICKERS` 就能換標的（yfinance 代碼：台股加 `.TW`，例如 0050→`0050.TW`）。

In [ ]:
import os, sys
sys.path.insert(0, ".")
from ve import fetch

TICKERS = {
    "ETF_0050":   "0050.TW",   # 元大台灣50
    "ETF_0056":   "0056.TW",   # 元大高股息
    "TSMC_2330":  "2330.TW",   # 台積電
    "MTK_2454":   "2454.TW",   # 聯發科
}

os.makedirs("data/real", exist_ok=True)
for name, tk in TICKERS.items():
    try:
        df = fetch.fetch_yfinance(tk, start="2015-01-01")
        df.to_csv(f"data/real/{name}.csv")
        print(f"  {name:12s} {tk:10s} {len(df)} 天  {df.index.min().date()} ~ {df.index.max().date()}")
    except Exception as e:
        print(f"  {name}: 抓取失敗 -> {e}")

## 3. 跑完整 VE（第一~三階段 + 成功標準）

In [ ]:
!python scripts/run_full.py --data data/real

## 4. 顯示圖表（在手機上直接看）

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("output/*.png")):
    print(f)
    display(Image(f))

## 5. 單一標的：現在在哪個波動階段？

In [ ]:
!python scripts/current_phase.py data/real/ETF_0050.csv

---
**提醒**：VE v0.2 的目的是建立能「預測下一階段」的波動生命週期模型。Phase ≠ Trade——判讀階段不是買賣訊號。只有轉換模型具備 out-of-sample 預測力後，才設計交易規則。

In [ ]:
!python scripts/run_transition_study.py --data data/real --horizon 5

## 7. 打假驗證：這個預測力是真的還是機械假象？

轉換 AUC 高，但**階段本身就是用波動特徵定義的**，可能有循環論證。這一格用四個對照組拆解：

* **placebo**（打亂標籤）→ 應該掉到 ~0.50（證明沒作弊）
* **time_only**（只用時間）→ 純「循環有固定長度」貢獻多少
* **external_target**（預測「未來 10 天實際波動會不會變高」，**非我們自己定義的階段**）→ 這個若還 >0.6，才是**真的**預測力

看最後三行的解讀：只有 external_target 撐得住、placebo 夠低，這個模型才值得信任。

In [ ]:
!python scripts/run_robustness.py --data data/real --horizon 5

## 8. 六層驗證框架（v0.2 完整驗證）

把 VE 從「高 AUC 分類器」推向「真正的理論」，逐層打假：

* **Layer 3 普遍性** — Cross-Time（跨年代穩定）、Cross-Asset（跨資產轉移）、Regime（牛熊盤）
* **Layer 4 新資訊** — Information Gain：新波動特徵有沒有超過 `{時間, HV百分位}` 基準
* **Layer 5 交易價值** — **Oracle Test（最重要）**：就算全知者，這個階段能不能賺錢？
  看 `long_only`（方向價值）vs `straddle_net`（波動價值）
* **Layer 6 機制** — Liquidity：流動性是否**領先**波動（VE 會不會其實是流動性生命週期？）

最終問題：**是否存在一種可跨時間、跨資產、跨環境重現，且具交易價值的波動生命週期？**

In [ ]:
!python scripts/run_validation.py --data data/real --horizon 5

---
**提醒**：這是 Phase-1 波動階段的位置判讀，不是買賣訊號。Build/Exit 交易層仍需用真實資料驗證與調參（見 README 第 11 點成功標準說明）。